# Imports 

In [1]:
import sys
print("Python version:", sys.version)

Python version: 3.10.6 | packaged by conda-forge | (main, Aug 22 2022, 20:36:39) [GCC 10.4.0]


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

print("Pytorch GPU loaded") if torch.cuda.is_available() else print("Pytorch CPU loaded!")

from torchvision import transforms as T

import timm

Pytorch GPU loaded


/home/genesis/CS/radio-signal-classifier/.pixi/envs/default/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


![](Untitled-design.png)

# Configurations

In [3]:
from pathlib import Path

TRAIN_CSV = Path("data/train.csv")
VALID_CSV = Path("data/valid.csv")

In [4]:
BATCH_SIZE = 128
DEVICE = "gpu"

In [5]:
MODEL_NAME = "efficientnet_b0"
LR = 0.001
EPOCHS = 15

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

In [ ]:
df_valid = pd.read_csv(VALID_CSV)
df_valid.head()

In [ ]:
print(f"No. of examples present in df_train : {len(df_train)}")
print(f"No. of examples present in df_valid : {len(df_valid)}")
print(f"Labels are : {df_train['labels'].unique()}")

In [ ]:
# Converting a sample image (presented as a row) into 2d representation
idx = 2

img_pixels = np.array(df_train.iloc[idx, 0:-2], dtype=np.float64) # Last column is label
img_label = df_train.iloc[idx, -1]

img_grid = np.resize(img_pixels, (64, 128)) # 64 * 128 = 8192

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img_grid)
ax.set_title(img_label)
ax.axis("off")
plt.show()

# Declare Spec Augmentations 

![](image6.png)

Vertical blank strips of noise => **TIME MASKS**
Horizontal strips => **FREQUENCY MASKS**

In [ ]:
from spec_augment import TimeMask, FreqMask

In [ ]:
# To compose a set of random transformations to be applied on an image dataset, we use pytorch's `torchvision.transform.Compose([...])` object

def get_train_transform():
    return T.Compose([
        TimeMask(T=15, num_masks=4), # T is width of VERTICAL mask here
        FreqMask(F=15, num_masks=3), # F is width of HORIZONTAL mask here
    ])

# Create Custom Dataset 

In [ ]:
class SpecDataset(Dataset):
    """
    Class inheriting from PyTorch's torch.utils.data.Dataset class to build Spectogram augmentations & visualisations easily.
    Image must be 64 * 128!
    """
    def __init__(self, df: pd.DataFrame, augmentations=None):
        self.df = df.copy(deep=True)
        self.augmentations=augmentations
        # label_encoding
        label_mapper = {
            "Squiggle": 0,
            "Narrowband": 1,
            "Narrowbanddrd": 2,
            "Noises": 3
        }
        self.df.iloc[:, -1] = self.df.iloc[:, -1].map(label_mapper)

    def __len__(self) -> int:
        return len(self.df)

    def head(self) -> pd.DataFrame:
        return self.df.head()

    def __getitem__(self, idx: int):
        """
        VERY IMPORTANT: This converts 1D pandas Series records into 2D spectrogram images!
        """
        row: pd.Series = self.df.iloc[idx]
        
        img_pixels = np.array(row.iloc[0:-2], dtype=np.float64) # Last column is label
        img_label = np.array(row.iloc[-1], dtype=np.int64)
        
        img_grid = np.resize(img_pixels, (64, 128, 1)) # (height, width, channel)

        # Converting to a PyTorch tensor with dims (channel, height, width), as this is the format with PyTorch
        img = torch.Tensor(img_grid).permute(2, 0, 1)

        if self.augmentations != None:
            img = self.augmentations(img)
        
        return img.float(), img_label

In [ ]:
trainset = SpecDataset(df_train, augmentations=get_train_transform())
validset = SpecDataset(df_valid, augmentations=None)

In [ ]:
# TESTING trainset output - Everytime we run this cell, new augmentation applied!
image, label = trainset[591]

plt.imshow(image.permute(0, 1, 2).squeeze())
plt.title(f"Label: {label}")
plt.show()

# Load dataset into Batches

In [ ]:
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
validloader = DataLoader(validset, batch_size=BATCH_SIZE)

In [ ]:
print(f"Total no. of batches in trainloader : {len(trainloader)}")
print(f"Total no. of batches in validloader : {len(validloader)}")

In [29]:
for images, labels in trainloader:
    break

print(f"One image batch shape : {images.shape}")
print(f"One label batch shape : {labels.shape}")

One image batch shape : torch.Size([128, 1, 64, 128])
One label batch shape : torch.Size([128])


# Load Model

# Create Train and Eval Function

In [ ]:
from tqdm.notebook import tqdm
from utils import multiclass_accuracy

# Training Loop 

# Inference 

In [4]:
from utils import view_classify